In [3]:
from nltk.tree import Tree

import pandas as pd

from nltk.corpus.reader.wordnet import Lemma
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
import numpy as np


from scipy.stats import spearmanr, pearsonr



In [8]:
# load semcor
tokens_df = pd.read_csv('/home/gsc685/data/semcor_all_tokens.csv')
tokens_df

,lemma,sense,word_form,sentence_id,pos
0,napoleon,Lemma('napoleon.n.01.Napoleon'),napoleon,13864,NN
1,wa,Lemma('be.v.01.be'),was,13864,VB
2,worst,Lemma('worst.a.01.worst'),worst,13864,JJ
3,loser,Lemma('loser.n.01.loser'),losers,13864,NN
4,island,Lemma('island.n.01.island'),islands,12747,NN
...,...,...,...,...,...
242636,ambiguity,Lemma('ambiguity.n.01.ambiguity'),ambiguities,10655,NN
242637,editorial,Lemma('column.n.05.editorial'),editorial,10655,NN
242638,advertising,Lemma('ad.n.01.advertising'),advertising,10655,NN
242639,matter,Lemma('topic.n.02.matter'),matter,10655,NN


In [3]:
# get 
tokens_df.groupby(['pos', 'lemma']).size().reset_index(name='count').sort_values('count', ascending=False)

,pos,lemma,count
26617,VB,is,5154
31006,VB,wa,4089
22821,VB,be,2181
22585,VB,are,1884
21625,RB,not,1700
...,...,...,...
22860,VB,beget,1
9710,NN,hater,1
9704,NN,hasp,1
22865,VB,begotten,1


In [4]:
# what are our most polysemous lemmas?
polysemes_df = tokens_df.groupby(['pos', 'lemma'])['sense'].nunique().reset_index()
polysemes_df

,pos,lemma,sense
0,JJ,0,1
1,JJ,1,2
2,JJ,1 2,1
3,JJ,1.0,1
4,JJ,1.5,1
...,...,...,...
31358,VB,zoomed,1
31359,VB,zooming,1
31360,VBD,wa,1
31361,VBG,being,1


In [5]:
# get a list of the most polysemous lemmas to keep out
polysemes = polysemes_df[polysemes_df["sense"] > 15]
polysemes.head(20)

,pos,lemma,sense
10911,NN,line,22
12719,NN,place,16
23102,VB,break,18
23423,VB,caught,17
23710,VB,come,17
24212,VB,cut,16
25722,VB,gave,23
25746,VB,get,26
25803,VB,give,29
25812,VB,given,23


In [23]:
# now pick 20 random words that arent too common or too uncommon
z = tokens_df[tokens_df['pos'].isin(['JJ', 'NN', 'VB'])]
z = z.groupby(['pos', 'lemma']).size().reset_index(name='count', level='lemma')
z = z[z['count'] > 50]
z = z[z['count'] < 500 ]
z = z.groupby('pos').sample(10)
#.apply(lambda x: x.sample(n=min(len(x),10), random_state=42)).reset_index()
z

,lemma,count
pos,,
JJ,no,90
JJ,first,283
JJ,one,406
JJ,third,71
JJ,large,122
JJ,high,143
JJ,clear,52
JJ,same,239
JJ,general,89


In [24]:
z.to_csv('/home/gsc685/data/validation_experiment_lemmas.csv')

In [29]:
# now i want to make a big tokens ccsv file with the sentence index of the token

validation_lemmas = pd.read_csv('/home/gsc685/data/validation_experiment_lemmas.csv')
lemma_list = validation_lemmas['lemma'].tolist()
lemma_list

['no',
 'first',
 'one',
 'third',
 'large',
 'high',
 'clear',
 'same',
 'general',
 'ready',
 'age',
 'information',
 'word',
 'door',
 'meaning',
 'government',
 'study',
 'animal',
 'growth',
 'building',
 'left',
 'seem',
 'died',
 'obtained',
 'ran',
 'built',
 'considered',
 'took',
 'stand',
 'suppose']

In [31]:
# Filter the DataFrame to keep only the selected lemmas
validation_tokens = tokens_df[tokens_df['lemma'].isin(lemma_list)]
len(validation_tokens)

4050

In [32]:
corpus = pd.read_csv('/home/gsc685/data/semcor_corpus.csv')
corpus.head()

merged = validation_tokens.merge(corpus, left_on='sentence_id', right_on='id', how='left')
validation_tokens = merged[merged.apply(lambda row: row['word_form'] in row['sentence'], axis=1)]
len(validation_tokens)

3858

In [33]:
validation_tokens = validation_tokens.drop(['sentence', 'id'], axis=1)
validation_tokens.head()

,lemma,sense,word_form,sentence_id,pos
0,left,Lemma('leave.v.03.leave'),left,25193,VB
1,obtained,Lemma('obtain.v.01.obtain'),obtained,32325,VB
2,same,Lemma('same.a.01.same'),same,2158,JJ
3,same,Lemma('same.a.02.same'),same,7738,JJ
4,general,Lemma('general.a.01.general'),general,16346,JJ


In [34]:
validation_tokens.to_csv('/home/gsc685/data/collected_tokens/semcor/semcor_validation_tokens.csv')
